In [1]:
import os
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import time
import seaborn as sns  # optional, just for color if you like
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import statsmodels.formula.api as smf

In [14]:
base_dir = "/Users/seohyon/resources/results"

keep_methods = [
    "embed_cell_types_jittered",
    "embed_cell_types",
    "shuffle_integration",
    "shuffle_integration_by_cell_type",
    "shuffle_integration",
    "scanvi",
    "combat",
    "scvi",
    "harmony",
    "harmonypy",
    "batchelor_fastmnn",
    "liger",
    "pyliger",
    "scalex",
    "uce",
    "no_integration",
    "no_integration_batch",
    "scanorama"
]

dfs_by_folder = {}   # <--- NEW: store one DF per folder

# Step 1 — Collect rows per folder
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    yaml_path = os.path.join(folder_path, "score_uns.yaml")

    # only process valid run folders
    if not (os.path.isdir(folder_path) and os.path.exists(yaml_path)):
        continue

    folder_rows = []     # rows only for this folder

    with open(yaml_path, "r") as f:
        try:
            data = yaml.safe_load(f)
        except yaml.YAMLError as e:
            print(f"⚠️ Could not read {yaml_path}: {e}")
            continue

    for entry in data:
        method = entry.get("method_id", "").strip().lower()
        if method not in keep_methods:
            continue

        metrics = entry.get("metric_ids", [])
        values = entry.get("metric_values", [])

        for m_id, m_val in zip(metrics, values):
            folder_rows.append({
                "dataset_id": entry.get("dataset_id"),
                "file_size": entry.get("file_size"),
                "method_id": method,
                "normalization_id": entry.get("normalization_id"),
                "metric_id": m_id,
                "metric_value": m_val,
                "run_folder": folder,     # optional: keep source folder
            })

    # Step 2 — create DataFrame for this folder
    df_folder = pd.DataFrame(folder_rows)
    df_folder = df_folder.dropna(subset=["metric_value"]).copy()
    df_folder = df_folder[~df_folder["metric_id"].isin(["kbet", "hvg_overlap"])].copy() # drop unwanted metrics
    df_folder["dataset_id"] = df_folder["dataset_id"].str.split("/").str[-1] # clean the dataset id

    dfs_by_folder[folder] = df_folder   # store it

dfs_by_folder = dict(sorted(dfs_by_folder.items())) # sort by date


In [15]:
for name, df_run in dfs_by_folder.items():
    print(name, df_run.groupby("metric_id").size())


run_2025-01-11_14-07-39 metric_id
ari                        93
asw_batch                  93
asw_label                  89
cell_cycle_conservation    45
clisi                      92
graph_connectivity         93
ilisi                      92
isolated_label_asw         91
isolated_label_f1          76
nmi                        93
pcr                        91
dtype: int64
run_2025-01-20_15-57-21 metric_id
ari                        92
asw_batch                  92
asw_label                  87
cell_cycle_conservation    88
clisi                      91
graph_connectivity         93
ilisi                      91
isolated_label_asw         66
isolated_label_f1          76
nmi                        92
pcr                        93
dtype: int64
run_2025-01-21_17-26-56 metric_id
ari                        89
asw_batch                  89
asw_label                  82
cell_cycle_conservation    89
clisi                      86
graph_connectivity         89
ilisi                      86
is

In [25]:
for name, df_run in dfs_by_folder.items():
    print(name, df_run.groupby(["metric_id", "method_id"]).size().unique())

run_2025-01-11_14-07-39 [6 5 1 4 3 2]
run_2025-01-20_15-57-21 [6 5 4 1 2 3]
run_2025-01-21_17-26-56 [6 4 5 2 3 1]
run_2025-01-23_18-03-16 [6 5]
run_2025-07-28_16-56-15 [6 5]
run_2025-07-29_12-58-29 [6 5]
run_2025-08-13_18-16-48 [6 5 4]
run_2025-10-11_13-28-50 [6 5]


In [28]:
for name, df_run in dfs_by_folder.items():
    print(f"\n===== {name} =====")
    group_df = (
        df_run.groupby(["metric_id", "method_id"])
              .size()
              .reset_index(name="count")
              .sort_values(["metric_id", "method_id"])
    )
    print(group_df.to_string(index=False))


===== run_2025-01-11_14-07-39 =====
              metric_id                        method_id  count
                    ari                batchelor_fastmnn      6
                    ari                           combat      6
                    ari                 embed_cell_types      6
                    ari        embed_cell_types_jittered      6
                    ari                          harmony      6
                    ari                        harmonypy      6
                    ari                            liger      5
                    ari                   no_integration      6
                    ari             no_integration_batch      5
                    ari                          pyliger      6
                    ari                           scalex      5
                    ari                        scanorama      5
                    ari                           scanvi      6
                    ari                             scvi      6
   

In [31]:
for name, df_run in dfs_by_folder.items():
    print(f"\n===== {name} =====")
    grouped = df_run.groupby("metric_id")["method_id"].unique()

    for metric, methods in grouped.items():
        print(metric, len(list(methods)))



===== run_2025-01-11_14-07-39 =====
ari 17
asw_batch 17
asw_label 16
cell_cycle_conservation 16
clisi 17
graph_connectivity 17
ilisi 17
isolated_label_asw 17
isolated_label_f1 16
nmi 17
pcr 17

===== run_2025-01-20_15-57-21 =====
ari 17
asw_batch 17
asw_label 17
cell_cycle_conservation 17
clisi 17
graph_connectivity 17
ilisi 17
isolated_label_asw 16
isolated_label_f1 16
nmi 17
pcr 17

===== run_2025-01-21_17-26-56 =====
ari 16
asw_batch 16
asw_label 16
cell_cycle_conservation 16
clisi 16
graph_connectivity 16
ilisi 16
isolated_label_asw 16
isolated_label_f1 16
nmi 16
pcr 16

===== run_2025-01-23_18-03-16 =====
ari 17
asw_batch 17
asw_label 17
cell_cycle_conservation 17
clisi 17
graph_connectivity 17
ilisi 17
isolated_label_asw 17
isolated_label_f1 17
nmi 17
pcr 17

===== run_2025-07-28_16-56-15 =====
ari 3
asw_batch 3
asw_label 3
cell_cycle_conservation 3
clisi 3
graph_connectivity 3
ilisi 3
isolated_label_asw 3
isolated_label_f1 3
kbet_pg 3
kbet_pg_label 3
nmi 3
pcr 3

===== run_2025

In [35]:
for name, df_run in dfs_by_folder.items():
    print(f"\n===== {name} =====")
    grouped = df_run.groupby("metric_id")["dataset_id"].unique()

    for metric, datasets in grouped.items():
        print(metric, len(list(datasets)))


===== run_2025-01-11_14-07-39 =====
ari 6
asw_batch 6
asw_label 6
cell_cycle_conservation 3
clisi 6
graph_connectivity 6
ilisi 6
isolated_label_asw 6
isolated_label_f1 5
nmi 6
pcr 6

===== run_2025-01-20_15-57-21 =====
ari 6
asw_batch 6
asw_label 6
cell_cycle_conservation 6
clisi 6
graph_connectivity 6
ilisi 6
isolated_label_asw 5
isolated_label_f1 5
nmi 6
pcr 6

===== run_2025-01-21_17-26-56 =====
ari 6
asw_batch 6
asw_label 6
cell_cycle_conservation 6
clisi 6
graph_connectivity 6
ilisi 6
isolated_label_asw 5
isolated_label_f1 5
nmi 6
pcr 6

===== run_2025-01-23_18-03-16 =====
ari 6
asw_batch 6
asw_label 6
cell_cycle_conservation 6
clisi 6
graph_connectivity 6
ilisi 6
isolated_label_asw 5
isolated_label_f1 5
nmi 6
pcr 6

===== run_2025-07-28_16-56-15 =====
ari 6
asw_batch 6
asw_label 6
cell_cycle_conservation 6
clisi 6
graph_connectivity 6
ilisi 6
isolated_label_asw 5
isolated_label_f1 5
kbet_pg 6
kbet_pg_label 6
nmi 6
pcr 6

===== run_2025-07-29_12-58-29 =====
ari 6
asw_batch 6
asw_

In [45]:
dfs_by_folder["run_2025-01-11_14-07-39"].query("metric_id == 'cell_cycle_conservation'")["dataset_id"].unique()

array(['immune_cell_atlas', 'tabula_sapiens', 'mouse_pancreas_atlas'],
      dtype=object)

In [41]:
for name, df_run in dfs_by_folder.items():
    print(f"\n===== {name} =====")
    filtered = df_run.query("metric_id in ['isolated_label_asw', 'isolated_label_f1']")
    grouped = filtered.groupby("metric_id")["dataset_id"].unique()


    for metric, datasets in grouped.items():
        print(metric, sorted(list(datasets)))


===== run_2025-01-11_14-07-39 =====
isolated_label_asw ['dkd', 'gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']
isolated_label_f1 ['gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']

===== run_2025-01-20_15-57-21 =====
isolated_label_asw ['gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']
isolated_label_f1 ['gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']

===== run_2025-01-21_17-26-56 =====
isolated_label_asw ['gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']
isolated_label_f1 ['gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']

===== run_2025-01-23_18-03-16 =====
isolated_label_asw ['gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']
isolated_label_f1 ['gtex_v9', 'hypomap', 'immune_cell_atlas', 'mouse_pancreas_atlas', 'tabula_sapiens']

===== ru

In [46]:
big_df = pd.concat(dfs_by_folder.values(), axis=0, ignore_index=True)

In [54]:
# Count per metric + method + dataset
group_counts = (
    big_df.groupby(["metric_id", "method_id", "dataset_id"])
      .size()
      .reset_index(name="count")
)

# Compute min count *per metric_id + method_id* 
group_counts["min_count"] = (
    group_counts
    .groupby(["metric_id", "method_id"])["count"]
    .transform("min")
)

per_method_min = (
    group_counts[["metric_id", "method_id", "min_count"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(per_method_min.to_string())


                   metric_id                         method_id  min_count
0                        ari                 batchelor_fastmnn          6
1                        ari                            combat          6
2                        ari                  embed_cell_types          7
3                        ari         embed_cell_types_jittered          6
4                        ari                           harmony          5
5                        ari                         harmonypy          8
6                        ari                             liger          5
7                        ari                    no_integration          7
8                        ari              no_integration_batch          6
9                        ari                           pyliger          4
10                       ari                            scalex          4
11                       ari                         scanorama          7
12                       ari          

In [55]:
per_method_min["min_count"].unique()

array([6, 7, 5, 8, 4, 3, 1, 2])

In [77]:
filtered_df = (
    group_counts
    .query("min_count >= 5")
    .reset_index(drop=True)
)# filtered_df now contains only metric–method pairs where every dataset has ≥ 5 datapoints.

In [80]:
counts = (
    filtered_df
    .groupby(["metric_id", "method_id", "dataset_id"])
    .size()
    .reset_index(name="count")
)

counts

,metric_id,method_id,dataset_id,count
0,ari,batchelor_fastmnn,dkd,1
1,ari,batchelor_fastmnn,gtex_v9,1
2,ari,batchelor_fastmnn,hypomap,1
3,ari,batchelor_fastmnn,immune_cell_atlas,1
4,ari,batchelor_fastmnn,mouse_pancreas_atlas,1
...,...,...,...,...
803,pcr,shuffle_integration_by_cell_type,gtex_v9,1
804,pcr,shuffle_integration_by_cell_type,hypomap,1
805,pcr,shuffle_integration_by_cell_type,immune_cell_atlas,1
806,pcr,shuffle_integration_by_cell_type,mouse_pancreas_atlas,1


In [81]:
filtered_df[["metric_id", "method_id", "dataset_id", "count"]]

,metric_id,method_id,dataset_id,count
0,ari,batchelor_fastmnn,dkd,6
1,ari,batchelor_fastmnn,gtex_v9,6
2,ari,batchelor_fastmnn,hypomap,6
3,ari,batchelor_fastmnn,immune_cell_atlas,6
4,ari,batchelor_fastmnn,mouse_pancreas_atlas,6
...,...,...,...,...
803,pcr,shuffle_integration_by_cell_type,gtex_v9,7
804,pcr,shuffle_integration_by_cell_type,hypomap,7
805,pcr,shuffle_integration_by_cell_type,immune_cell_atlas,7
806,pcr,shuffle_integration_by_cell_type,mouse_pancreas_atlas,7


In [85]:
last_run_df = dfs_by_folder["run_2025-10-11_13-28-50"]
# Count per metric + method + dataset
group_counts = (
    last_run_df.groupby(["metric_id", "method_id", "dataset_id"])
      .size()
      .reset_index(name="count")
)

# Compute min count *per metric_id + method_id* 
group_counts["min_count"] = (
    group_counts
    .groupby(["metric_id", "method_id"])["count"]
    .transform("min")
)

last_run_df = (
    group_counts
    #.query("min_count >= 5")
    .reset_index(drop=True)
)
                                             
print(last_run_df[["metric_id", "method_id", "dataset_id", "count"]].to_string())

                    metric_id                         method_id            dataset_id  count
0                         ari                 batchelor_fastmnn                   dkd      1
1                         ari                 batchelor_fastmnn               gtex_v9      1
2                         ari                 batchelor_fastmnn               hypomap      1
3                         ari                 batchelor_fastmnn     immune_cell_atlas      1
4                         ari                 batchelor_fastmnn  mouse_pancreas_atlas      1
5                         ari                 batchelor_fastmnn        tabula_sapiens      1
6                         ari                            combat                   dkd      1
7                         ari                            combat               gtex_v9      1
8                         ari                            combat               hypomap      1
9                         ari                            combat     im

In [70]:
# 1. List all unique metrics
metrics_left = filtered_df["metric_id"].unique()

print("The metrics left in filtered_df:")
for m in metrics_left:
    print(f"  • {m}")

print("\nThe methods left for each metric:")

# 2. For each metric, list its methods
for metric in metrics_left:
    methods = (
        filtered_df.loc[filtered_df.metric_id == metric, "method_id"]
        .unique()
    )
    methods = sorted(methods)
    methods_list = ", ".join(methods)
    print(f"  • {metric}: {methods_list}")


The metrics left in filtered_df:
  • ari
  • asw_batch
  • asw_label
  • cell_cycle_conservation
  • clisi
  • graph_connectivity
  • ilisi
  • isolated_label_f1
  • nmi
  • pcr

The methods left for each metric:
  • ari: batchelor_fastmnn, combat, embed_cell_types, embed_cell_types_jittered, harmony, harmonypy, liger, no_integration, no_integration_batch, scanorama, scanvi, scvi, shuffle_integration, shuffle_integration_by_cell_type
  • asw_batch: batchelor_fastmnn, combat, embed_cell_types, embed_cell_types_jittered, harmony, harmonypy, liger, no_integration, no_integration_batch, scanorama, scanvi, scvi, shuffle_integration, shuffle_integration_by_cell_type
  • asw_label: batchelor_fastmnn, combat, embed_cell_types, embed_cell_types_jittered, harmony, harmonypy, liger, no_integration, no_integration_batch, scanorama, scvi, shuffle_integration, shuffle_integration_by_cell_type
  • cell_cycle_conservation: batchelor_fastmnn, combat, embed_cell_types, harmony, harmonypy, no_integration

In [73]:
# check if the methods left are same
methods_per_metric = (
    filtered_df.groupby("metric_id")["method_id"]
    .apply(lambda x: frozenset(x.unique()))
)

# Are they all identical?
all_same = methods_per_metric.nunique() == 1

print("Do all metrics have the same methods left?:", all_same)

Do all metrics have the same methods left?: False


In [75]:
# Convert frozensets → sets for intersection math
method_sets = list(map(set, methods_per_metric))

# ---- 1️⃣ Methods in ALL metrics ----
methods_in_all_metrics = set.intersection(*method_sets)

print("Methods present in ALL metrics:")
print(sorted(methods_in_all_metrics))

# ---- 2️⃣ Methods in ONLY SOME metrics ----
methods_in_any_metric = set.union(*method_sets)
methods_in_some_metrics = methods_in_any_metric - methods_in_all_metrics

print("\nMethods present in ONLY SOME metrics:")
print(sorted(methods_in_some_metrics))

Methods present in ALL metrics:
['batchelor_fastmnn', 'combat', 'embed_cell_types', 'harmony', 'harmonypy', 'no_integration', 'no_integration_batch', 'scanorama', 'scvi', 'shuffle_integration', 'shuffle_integration_by_cell_type']

Methods present in ONLY SOME metrics:
['embed_cell_types_jittered', 'liger', 'scanvi']
